In [ ]:
# Contenuti in evidenza della cartella
# Esercizi di Machine Learning, incluso l'Esercizio 10 dettagliato su classificazione immagini con MobileNetV2 e correzione errori.

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
ESERCIZIO 10 - Classificazione Immagini: Cappello vs Scarpa
================================================================================

PROMPT STRUTTURATO UTILIZZATO:
─────────────────────────────
"Crea un sistema di classificazione di immagini con Python e Keras che:
1. Generi almeno 200 immagini per classe (cappello e scarpa) tramite data augmentation
   partendo da una singola immagine per classe
2. Utilizzi transfer learning con MobileNetV2 pre-addestrato su ImageNet
3. Implementi early stopping per prevenire l'overfitting
4. Raggiunga almeno l'80% di accuratezza
5. Salvi i parametri della rete in un file classi.pkl
6. Mostri metriche di valutazione complete con grafici e matrici di confusione
7. Sia ottimizzato per Google Colab con GPU"

COMPONENTI DELLA RETE NEURALE:
─────────────────────────────
1. INPUT LAYER: accetta immagini 224x224x3 (RGB)
2. BASE MODEL: MobileNetV2 pre-addestrato (transfer learning) - pesi congelati
3. GLOBAL AVERAGE POOLING: riduce le feature map a un vettore 1D
4. DROPOUT LAYER (0.3): previene overfitting spegnendo casualmente il 30% dei neuroni
5. DENSE LAYER (128 neuroni, ReLU): livello nascosto per apprendere pattern complessi
6. DROPOUT LAYER (0.2): ulteriore regolarizzazione
7. OUTPUT LAYER (1 neurone, Sigmoid): classificazione binaria (0=cappello, 1=scarpa)

PARAMETRI SCELTI E MOTIVAZIONI:
──────────────────────────────
- MobileNetV2: modello leggero ma efficace, ideale per transfer learning con pochi dati
- Learning Rate 0.0001: basso per non distruggere i pesi pre-addestrati
- Batch Size 16: piccolo perché abbiamo poche immagini (evita generalizzazione prematura)
- Image Size 224x224: dimensione standard per MobileNetV2
- Dropout 0.3 + 0.2: regolarizzazione progressiva contro overfitting
- Dense 128: compromesso tra capacità di apprendimento e rischio overfitting
- Sigmoid + Binary Crossentropy: standard per classificazione binaria
- Adam optimizer: adattivo, converge bene con pochi dati
- Early Stopping patience=10: dà tempo sufficiente per trovare il minimo

COME OTTIMIZZARE I PARAMETRI:
────────────────────────────
1. Se l'accuratezza è bassa (<80%):
   - Aumentare il numero di immagini augmentate (es. 400+)
   - Sbloccare gli ultimi livelli di MobileNetV2 (fine-tuning)
   - Aumentare i neuroni del Dense layer (es. 256)
   - Ridurre il dropout (es. 0.15)
2. Se c'è overfitting (val_loss sale mentre train_loss scende):
   - Aumentare il dropout (es. 0.4-0.5)
   - Aggiungere più data augmentation (shear, zoom, channel_shift)
   - Ridurre il numero di neuroni Dense
   - Diminuire il learning rate
3. Se l'addestramento è lento:
   - Aumentare il batch size (32, 64)
   - Usare learning rate scheduling (ReduceLROnPlateau)
4. Se le classi sono sbilanciate:
   - Usare class_weight nel fit()
   - Applicare oversampling della classe minoritaria

ERRORI SCOPERTI DURANTE IL TEST:
───────────────────────────────
- Errore 1: Percorso file immagini originali non corretto → corretto con os.path.join
- Errore 2: Mancava la normalizzazione preprocessing_input per MobileNetV2 → aggiunto
- Errore 3: Shape mismatch nel modello → corretto input_shape a (224,224,3)
- Errore 4: Classi sbilanciate nel dataset → aggiunto bilanciamento con class_weight
- Errore 5: Salvataggio classi.pkl mancava le label → aggiunto dizionario completo

TEMPISTICHE:
───────────
Inizio: 12:00 - Fine: 12:45
L'addestramento richiede ~5-10 minuti su Google Colab con GPU

================================================================================
"""

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 0: INSTALLAZIONE E SETUP PER GOOGLE COLAB                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# --- Se esegui su Google Colab, decommenta queste righe: ---
# !pip install tensorflow keras scikit-learn matplotlib seaborn pillow

# --- Verifica che la GPU sia attiva su Colab ---
# In Colab: Runtime → Change runtime type → GPU
# import tensorflow as tf
# print("GPU disponibile:", tf.config.list_physical_devices('GPU'))

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 1: IMPORTAZIONE LIBRERIE                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os                          # Per gestire file e cartelle
import numpy as np                 # Per calcoli numerici e array
import pickle                      # Per salvare/caricare oggetti Python (classi.pkl)
import warnings                    # Per silenziare avvisi non critici
warnings.filterwarnings('ignore')

# --- Librerie per le immagini ---
from PIL import Image              # Per manipolare le immagini (apertura, ridimensionamento)

# --- TensorFlow e Keras (il motore della rete neurale) ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,           # Genera immagini augmentate in tempo reale
    img_to_array,                 # Converte immagine PIL in array numpy
    load_img                      # Carica un'immagine da file
)
from tensorflow.keras.applications import MobileNetV2                # Modello pre-addestrato
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input  # Preprocessing specifico
from tensorflow.keras.models import Model, Sequential                # Per costruire il modello
from tensorflow.keras.layers import (
    Dense,                        # Livello completamente connesso (ogni neurone connesso a tutti)
    Dropout,                      # Spegne casualmente dei neuroni (previene overfitting)
    GlobalAveragePooling2D,       # Riduce le feature map a un vettore
    Input                         # Definisce l'input del modello
)
from tensorflow.keras.optimizers import Adam                         # Ottimizzatore adattivo
from tensorflow.keras.callbacks import (
    EarlyStopping,                # Ferma l'addestramento se non migliora (contro overfitting)
    ModelCheckpoint,              # Salva il modello migliore durante l'addestramento
    ReduceLROnPlateau             # Riduce il learning rate se l'apprendimento si blocca
)
from tensorflow.keras.utils import to_categorical                    # One-hot encoding delle label

# --- Librerie per valutazione e grafici ---
from sklearn.model_selection import train_test_split                 # Divide dati in train/test
from sklearn.metrics import (
    classification_report,        # Report dettagliato: precision, recall, F1-score
    confusion_matrix,             # Matrice di confusione
    accuracy_score,               # Accuratezza complessiva
    precision_recall_fscore_support,  # Metriche per classe
    roc_curve,                    # Curva ROC
    auc                           # Area Under Curve
)
import matplotlib.pyplot as plt   # Per creare grafici
import seaborn as sns             # Per grafici più belli (matrice di confusione)

# --- Verifica GPU ---
print("=" * 60)
print("SETUP AMBIENTE")
print("=" * 60)
print(f"TensorFlow versione: {tf.__version__}")
print(f"GPU disponibili: {tf.config.list_physical_devices('GPU')}")
if tf.config.list_physical_devices('GPU'):
    print("✅ GPU attiva! L'addestramento sarà veloce.")
else:
    print("⚠️  Nessuna GPU trovata. Su Colab: Runtime → Change runtime type → GPU")
print()


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 2: CONFIGURAZIONE PARAMETRI DELLA RETE NEURALE                    ║
# ║                                                                            ║
# ║ Qui definiamo tutti i parametri che controllano il comportamento della rete ║
# ║ Modificare questi valori per ottimizzare le prestazioni                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# --- Parametri delle immagini ---
IMG_SIZE = 224          # Dimensione in pixel (224x224 è lo standard per MobileNetV2)
                        # MobileNetV2 è stato addestrato con questa dimensione
IMG_CHANNELS = 3        # 3 canali = RGB (rosso, verde, blu)

# --- Parametri della data augmentation ---
# Quante immagini creare partendo da una singola foto
NUM_AUGMENTED_IMAGES = 250  # Creiamo 250 per classe (minimo richiesto: 200)

# --- Parametri di addestramento ---
BATCH_SIZE = 16         # Quante immagini elaborare contemporaneamente
                        # Piccolo = più preciso ma più lento
                        # Grande = più veloce ma meno preciso
                        # 16 è un buon compromesso per dataset piccoli

EPOCHS = 50             # Numero massimo di "giri" completi sul dataset
                        # L'early stopping fermerà prima se non migliora

LEARNING_RATE = 0.0001  # Velocità di apprendimento (quanto aggiorna i pesi)
                        # Basso (0.0001) perché usiamo transfer learning
                        # Se troppo alto: distrugge i pesi pre-addestrati
                        # Se troppo basso: non impara abbastanza

VALIDATION_SPLIT = 0.2  # 20% dei dati per validazione, 80% per addestramento

# --- Parametri del modello ---
DENSE_UNITS = 128       # Neuroni nel livello nascosto
                        # Più neuroni = più capacità di apprendimento
                        # Ma anche più rischio di overfitting
                        # 128 è un buon punto di partenza

DROPOUT_RATE_1 = 0.3    # Primo dropout: spegne il 30% dei neuroni
DROPOUT_RATE_2 = 0.2    # Secondo dropout: spegne il 20% dei neuroni
                        # Il dropout aiuta la rete a non "memorizzare" i dati

# --- Parametri early stopping ---
PATIENCE = 10           # Quante epoche aspettare senza miglioramento prima di fermarsi
                        # Troppo basso (3-5): potrebbe fermarsi prematuramente
                        # Troppo alto (20+): spreca tempo se è già in overfitting

# --- Nomi delle classi ---
CLASS_NAMES = ['cappello', 'scarpa']  # Le due categorie da classificare

# --- Percorsi file ---
BASE_DIR = './'                      # Cartella di lavoro
DATASET_DIR = os.path.join(BASE_DIR, 'dataset')  # Dove salvare le immagini augmentate
MODEL_SAVE_PATH = os.path.join(BASE_DIR, 'classi.pkl')  # File per salvare i parametri

# Stampa riepilogo configurazione
print("=" * 60)
print("CONFIGURAZIONE PARAMETRI")
print("=" * 60)
print(f"Dimensione immagini: {IMG_SIZE}x{IMG_SIZE}x{IMG_CHANNELS}")
print(f"Immagini augmentate per classe: {NUM_AUGMENTED_IMAGES}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epoche massime: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Neuroni Dense layer: {DENSE_UNITS}")
print(f"Dropout rates: {DROPOUT_RATE_1}, {DROPOUT_RATE_2}")
print(f"Early stopping patience: {PATIENCE}")
print()


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 3: DATA AUGMENTATION                                              ║
# ║                                                                            ║
# ║ La data augmentation crea nuove immagini a partire da una singola foto    ║
# ║ applicando trasformazioni casuali (rotazione, zoom, flip, ecc.)           ║
# ║ Questo è fondamentale quando abbiamo pochissime immagini originali        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def create_augmented_images(source_image_path, output_dir, class_name, num_images=250):
    """
    Crea immagini augmentate a partire da una singola immagine.

    COME FUNZIONA LA DATA AUGMENTATION:
    Prende un'immagine e applica trasformazioni casuali per creare
    varianti diverse. Ogni immagine generata è leggermente diversa
    dall'originale, il che aiuta la rete neurale a generalizzare.

    Parametri:
    ----------
    source_image_path : str
        Percorso dell'immagine originale (es. "image augmentation cappello.jpg")
    output_dir : str
        Cartella dove salvare le immagini generate
    class_name : str
        Nome della classe (es. "cappello" o "scarpa")
    num_images : int
        Quante immagini generare (default: 250)

    Ritorna:
    --------
    int : numero di immagini effettivamente create
    """

    # Crea la cartella di output se non esiste
    class_dir = os.path.join(output_dir, class_name)
    os.makedirs(class_dir, exist_ok=True)

    # Carica l'immagine originale e ridimensionala a 224x224
    print(f"\n📸 Caricamento immagine: {source_image_path}")
    img = load_img(source_image_path, target_size=(IMG_SIZE, IMG_SIZE))

    # Converte l'immagine in un array numpy (matrice di numeri)
    # Forma: (224, 224, 3) → 224 righe, 224 colonne, 3 canali colore
    img_array = img_to_array(img)

    # Aggiunge una dimensione per il batch: (1, 224, 224, 3)
    # ImageDataGenerator richiede un batch di immagini, anche se ne abbiamo una sola
    img_array = np.expand_dims(img_array, axis=0)

    # --- CONFIGURAZIONE DATA AUGMENTATION ---
    # Ogni parametro controlla un tipo di trasformazione casuale
    augmentation_generator = ImageDataGenerator(
        rotation_range=40,          # Ruota l'immagine fino a ±40 gradi
                                    # Simula oggetti visti da angolazioni diverse

        width_shift_range=0.2,      # Sposta orizzontalmente fino al 20% della larghezza
                                    # L'oggetto può essere decentrato

        height_shift_range=0.2,     # Sposta verticalmente fino al 20% dell'altezza
                                    # L'oggetto può essere più in alto o in basso

        shear_range=0.2,            # Deformazione a taglio (inclina l'immagine)
                                    # Simula prospettive diverse

        zoom_range=0.3,             # Zoom in/out fino al 30%
                                    # L'oggetto può apparire più grande o più piccolo

        horizontal_flip=True,       # Specchia l'immagine orizzontalmente
                                    # Un cappello visto da sinistra o da destra è sempre un cappello

        vertical_flip=False,        # NON specchiamo verticalmente
                                    # Un cappello capovolto non è realistico

        brightness_range=[0.7, 1.3],  # Varia la luminosità (70%-130%)
                                      # Simula diverse condizioni di luce

        channel_shift_range=30,     # Varia i canali colore
                                    # Simula diverse tonalità di colore

        fill_mode='nearest'         # Come riempire i pixel vuoti dopo la trasformazione
                                    # 'nearest' usa il pixel più vicino (risultato naturale)
    )

    # --- GENERAZIONE IMMAGINI ---
    print(f"🔄 Generazione {num_images} immagini augmentate per classe '{class_name}'...")

    # Crea il generatore di immagini
    # flow() genera immagini trasformate una alla volta
    generator = augmentation_generator.flow(
        img_array,                          # L'immagine originale
        batch_size=1,                       # Una immagine alla volta
        save_to_dir=class_dir,              # Dove salvare
        save_prefix=f'{class_name}_aug',    # Prefisso del nome file
        save_format='jpg'                   # Formato di salvataggio
    )

    # Genera le immagini una per una
    count = 0
    for _ in range(num_images):
        next(generator)  # Genera e salva una nuova immagine
        count += 1
        if count % 50 == 0:
            print(f"   → Generate {count}/{num_images} immagini...")

    print(f"✅ Create {count} immagini augmentate in: {class_dir}")
    return count


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 4: CARICAMENTO E PREPARAZIONE DEL DATASET                         ║
# ║                                                                            ║
# ║ Carica tutte le immagini generate, le preprocessa per MobileNetV2,        ║
# ║ crea le label (etichette) e divide in train/test                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def load_dataset(dataset_dir):
    """
    Carica tutte le immagini dal dataset e le prepara per la rete neurale.

    PREPROCESSING PER MOBILENETV2:
    MobileNetV2 si aspetta pixel nell'intervallo [-1, 1] (non 0-255).
    La funzione preprocess_input() di MobileNetV2 fa questa conversione.

    Ritorna:
    --------
    X : array numpy con tutte le immagini preprocessate
    y : array numpy con le label (0=cappello, 1=scarpa)
    """
    images = []     # Lista per le immagini (array di pixel)
    labels = []     # Lista per le etichette (0 o 1)

    print("\n" + "=" * 60)
    print("CARICAMENTO DATASET")
    print("=" * 60)

    # Scorre le classi (cappello=0, scarpa=1)
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(dataset_dir, class_name)

        if not os.path.exists(class_dir):
            print(f"⚠️  Cartella non trovata: {class_dir}")
            continue

        # Conta le immagini nella cartella
        image_files = [f for f in os.listdir(class_dir)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        print(f"\n📂 Classe '{class_name}' (label={class_idx}): {len(image_files)} immagini")

        # Carica ogni immagine
        for i, filename in enumerate(image_files):
            filepath = os.path.join(class_dir, filename)
            try:
                # Carica e ridimensiona l'immagine
                img = load_img(filepath, target_size=(IMG_SIZE, IMG_SIZE))
                img_array = img_to_array(img)

                # Applica il preprocessing di MobileNetV2
                # Converte i pixel da [0, 255] a [-1, 1]
                img_array = preprocess_input(img_array)

                images.append(img_array)
                labels.append(class_idx)

            except Exception as e:
                print(f"   ⚠️  Errore caricamento {filename}: {e}")
                continue

            if (i + 1) % 50 == 0:
                print(f"   → Caricate {i + 1}/{len(image_files)} immagini...")

    # Converte le liste in array numpy
    X = np.array(images)
    y = np.array(labels)

    print(f"\n📊 Dataset totale:")
    print(f"   Immagini: {X.shape[0]}")
    print(f"   Dimensione: {X.shape[1:]} (altezza, larghezza, canali)")
    print(f"   Cappelli: {np.sum(y == 0)}")
    print(f"   Scarpe: {np.sum(y == 1)}")

    # SOSPETTO: Verifica bilanciamento classi
    if np.sum(y == 0) != np.sum(y == 1):
        print(f"\n⚠️  ATTENZIONE: Classi sbilanciate!")
        print(f"   Cappelli: {np.sum(y == 0)} vs Scarpe: {np.sum(y == 1)}")
        print(f"   Verrà applicato class_weight per bilanciare l'addestramento")

    return X, y


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 5: COSTRUZIONE DEL MODELLO (RETE NEURALE)                         ║
# ║                                                                            ║
# ║ ARCHITETTURA CON TRANSFER LEARNING:                                        ║
# ║                                                                            ║
# ║ Input (224x224x3)                                                          ║
# ║      ↓                                                                     ║
# ║ MobileNetV2 (congelato)  ← Pesi pre-addestrati su ImageNet (1000 classi)  ║
# ║      ↓                     Riconosce bordi, forme, texture, pattern        ║
# ║ GlobalAveragePooling2D    ← Riduce le feature map a un vettore 1D         ║
# ║      ↓                                                                     ║
# ║ Dropout (0.3)             ← Spegne il 30% dei neuroni (anti-overfitting)  ║
# ║      ↓                                                                     ║
# ║ Dense (128, ReLU)         ← Livello nascosto, impara a combinare feature  ║
# ║      ↓                                                                     ║
# ║ Dropout (0.2)             ← Spegne il 20% dei neuroni (anti-overfitting)  ║
# ║      ↓                                                                     ║
# ║ Dense (1, Sigmoid)        ← Output: probabilità (0=cappello, 1=scarpa)    ║
# ║                                                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def build_model():
    """
    Costruisce il modello di classificazione con Transfer Learning.

    TRANSFER LEARNING:
    Invece di addestrare una rete da zero (servirebbero migliaia di immagini),
    usiamo MobileNetV2 che è stato già addestrato su ImageNet (1.4 milioni di
    immagini, 1000 classi). La rete ha già "imparato" a riconoscere forme,
    bordi, texture. Noi aggiungiamo solo i livelli finali per la nostra
    classificazione specifica (cappello vs scarpa).

    Ritorna:
    --------
    model : il modello Keras compilato e pronto per l'addestramento
    """

    print("\n" + "=" * 60)
    print("COSTRUZIONE MODELLO")
    print("=" * 60)

    # --- CARICAMENTO MOBILENETV2 PRE-ADDESTRATO ---
    # include_top=False: rimuove i livelli finali di classificazione di ImageNet
    # Vogliamo solo la parte che estrae le feature (caratteristiche) dalle immagini
    # weights='imagenet': usa i pesi pre-addestrati su ImageNet
    base_model = MobileNetV2(
        weights='imagenet',                     # Pesi pre-addestrati
        include_top=False,                      # Senza i livelli di classificazione originali
        input_shape=(IMG_SIZE, IMG_SIZE, IMG_CHANNELS)  # Dimensione input: 224x224x3
    )

    # --- CONGELAMENTO PESI ---
    # "Congela" i pesi del modello base: NON verranno modificati durante l'addestramento
    # Questo preserva ciò che MobileNetV2 ha già imparato
    # Addestriamo SOLO i nuovi livelli che aggiungiamo sopra
    base_model.trainable = False

    print(f"📦 MobileNetV2 caricato: {len(base_model.layers)} livelli congelati")

    # --- COSTRUZIONE DEL MODELLO COMPLETO ---
    # Aggiungiamo i nostri livelli di classificazione sopra MobileNetV2
    model = Sequential([
        # Livello di input
        Input(shape=(IMG_SIZE, IMG_SIZE, IMG_CHANNELS)),

        # MobileNetV2 come estrattore di feature
        # Prende l'immagine 224x224x3 e produce feature map 7x7x1280
        base_model,

        # Global Average Pooling: riduce 7x7x1280 → 1280
        # Media su ogni feature map, produce un vettore di 1280 numeri
        GlobalAveragePooling2D(),

        # Primo Dropout: spegne casualmente il 30% dei neuroni
        # PERCHÉ: con pochi dati, la rete tende a "memorizzare" le immagini
        # Il dropout la costringe a usare feature diverse ogni volta
        Dropout(DROPOUT_RATE_1),

        # Livello Dense (completamente connesso): 128 neuroni
        # Ogni neurone riceve input da tutti i 1280 valori precedenti
        # ReLU = Rectified Linear Unit: f(x) = max(0, x)
        # Attiva solo i neuroni con valore positivo (introduce non-linearità)
        Dense(DENSE_UNITS, activation='relu'),

        # Secondo Dropout: spegne il 20% dei neuroni
        # Ulteriore regolarizzazione prima dell'output
        Dropout(DROPOUT_RATE_2),

        # Livello Output: 1 neurone con Sigmoid
        # Sigmoid: produce un valore tra 0 e 1
        # Vicino a 0 = cappello, vicino a 1 = scarpa
        # Soglia 0.5: se > 0.5 → scarpa, se < 0.5 → cappello
        Dense(1, activation='sigmoid')
    ])

    # --- COMPILAZIONE DEL MODELLO ---
    # Definisce come il modello "impara"
    model.compile(
        # Adam: ottimizzatore adattivo
        # Regola automaticamente il learning rate per ogni parametro
        # È il più usato perché converge bene nella maggior parte dei casi
        optimizer=Adam(learning_rate=LEARNING_RATE),

        # Binary Crossentropy: funzione di perdita per classificazione binaria
        # Misura quanto le predizioni sono lontane dalla realtà
        # Più basso = predizioni migliori
        loss='binary_crossentropy',

        # Metriche da monitorare durante l'addestramento
        metrics=['accuracy']
    )

    # Stampa il riepilogo del modello
    print("\n📐 Architettura del modello:")
    model.summary()

    return model


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 6: ADDESTRAMENTO CON EARLY STOPPING                               ║
# ║                                                                            ║
# ║ EARLY STOPPING: ferma l'addestramento quando la rete smette di migliorare ║
# ║ sui dati di validazione. Previene l'overfitting (memorizzazione).          ║
# ║                                                                            ║
# ║ Senza early stopping:                                                      ║
# ║ Epoca 1-20: migliora su train E validation ✅                              ║
# ║ Epoca 20-50: migliora su train MA PEGGIORA su validation ❌ (overfitting)  ║
# ║                                                                            ║
# ║ Con early stopping:                                                        ║
# ║ Epoca 1-20: migliora su train E validation ✅                              ║
# ║ Epoca 20-30: validation non migliora → STOP! 🛑                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def train_model(model, X_train, y_train, X_val, y_val, class_weight=None):
    """
    Addestra il modello con early stopping e salvataggio del modello migliore.

    Parametri:
    ----------
    model : il modello Keras da addestrare
    X_train, y_train : dati di addestramento (immagini e label)
    X_val, y_val : dati di validazione (per monitorare overfitting)
    class_weight : pesi delle classi (per bilanciare classi sbilanciate)

    Ritorna:
    --------
    history : storico dell'addestramento (loss e accuracy per ogni epoca)
    """

    print("\n" + "=" * 60)
    print("ADDESTRAMENTO DEL MODELLO")
    print("=" * 60)

    # --- CALLBACKS: azioni automatiche durante l'addestramento ---
    callbacks = [
        # EARLY STOPPING: ferma l'addestramento se non migliora
        EarlyStopping(
            monitor='val_loss',          # Cosa monitorare: la loss sui dati di validazione
            patience=PATIENCE,           # Quante epoche aspettare senza miglioramento
            restore_best_weights=True,   # Ripristina i pesi dell'epoca migliore
            verbose=1                    # Mostra messaggi quando si attiva
        ),

        # REDUCE LR ON PLATEAU: riduce il learning rate quando si blocca
        # Se la val_loss non migliora per 5 epoche, dimezza il learning rate
        # Questo aiuta a "raffinare" l'apprendimento quando è vicino all'ottimo
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,                  # Moltiplica LR × 0.5 (dimezza)
            patience=5,                  # Aspetta 5 epoche senza miglioramento
            min_lr=1e-7,                 # Non scendere sotto 0.0000001
            verbose=1
        ),

        # MODEL CHECKPOINT: salva il modello migliore
        ModelCheckpoint(
            'best_model_Esercizio10.keras',  # Nome file
            monitor='val_accuracy',          # Salva quando l'accuratezza di validazione migliora
            save_best_only=True,             # Salva solo se è il migliore finora
            verbose=1
        )
    ]

    # --- Data augmentation DURANTE l'addestramento ---
    # Applica trasformazioni casuali alle immagini di training in tempo reale
    # Così la rete vede immagini sempre leggermente diverse
    train_datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        shear_range=0.1,
        zoom_range=0.15,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    print(f"\n🏋️ Inizio addestramento...")
    print(f"   Training samples: {len(X_train)}")
    print(f"   Validation samples: {len(X_val)}")
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Epoche massime: {EPOCHS}")
    print(f"   Early stopping patience: {PATIENCE}")
    if class_weight:
        print(f"   Class weights: {class_weight}")
    print()

    # --- ADDESTRAMENTO ---
    # fit() addestra il modello sui dati
    history = model.fit(
        train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
        epochs=EPOCHS,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1
    )

    # Risultati finali
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    epochs_trained = len(history.history['accuracy'])

    print(f"\n{'=' * 60}")
    print(f"RISULTATI ADDESTRAMENTO")
    print(f"{'=' * 60}")
    print(f"Epoche completate: {epochs_trained}/{EPOCHS}")
    print(f"Accuratezza training: {final_train_acc:.4f} ({final_train_acc*100:.1f}%)")
    print(f"Accuratezza validazione: {final_val_acc:.4f} ({final_val_acc*100:.1f}%)")

    if final_val_acc >= 0.80:
        print(f"✅ Obiettivo raggiunto! Accuratezza ≥ 80%")
    else:
        print(f"⚠️  Accuratezza sotto l'80%. Considerare ottimizzazione parametri.")

    return history


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 7: SALVATAGGIO PARAMETRI IN classi.pkl                             ║
# ║                                                                            ║
# ║ Salva tutto il necessario per riutilizzare il modello in futuro:          ║
# ║ - Nomi delle classi                                                        ║
# ║ - Parametri di configurazione                                              ║
# ║ - Metriche di addestramento                                                ║
# ║ Il modello vero e proprio viene salvato separatamente in formato .keras    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def save_model_params(model, history, class_names, filepath='classi.pkl'):
    """
    Salva i parametri della rete neurale in un file pickle.

    Il file classi.pkl contiene un dizionario con:
    - class_names: nomi delle classi ['cappello', 'scarpa']
    - class_indices: mappatura {0: 'cappello', 1: 'scarpa'}
    - img_size: dimensione delle immagini (224)
    - model_config: configurazione del modello
    - training_history: storico loss e accuracy
    - best_val_accuracy: miglior accuratezza raggiunta
    - parameters: tutti i parametri usati

    NOTA: i pesi del modello sono salvati in file .keras separato
    perché pickle non gestisce bene i tensori di TensorFlow
    """

    print("\n" + "=" * 60)
    print("SALVATAGGIO PARAMETRI")
    print("=" * 60)

    # Dizionario con tutti i parametri da salvare
    model_params = {
        # Informazioni sulle classi
        'class_names': class_names,
        'class_indices': {i: name for i, name in enumerate(class_names)},
        'num_classes': len(class_names),

        # Configurazione immagini
        'img_size': IMG_SIZE,
        'img_channels': IMG_CHANNELS,

        # Parametri del modello
        'parameters': {
            'model_name': 'MobileNetV2_TransferLearning',
            'dense_units': DENSE_UNITS,
            'dropout_rate_1': DROPOUT_RATE_1,
            'dropout_rate_2': DROPOUT_RATE_2,
            'learning_rate': LEARNING_RATE,
            'batch_size': BATCH_SIZE,
            'optimizer': 'Adam',
            'loss_function': 'binary_crossentropy',
            'output_activation': 'sigmoid',
        },

        # Storico addestramento
        'training_history': {
            'accuracy': history.history['accuracy'],
            'val_accuracy': history.history['val_accuracy'],
            'loss': history.history['loss'],
            'val_loss': history.history['val_loss'],
        },

        # Metriche finali
        'best_val_accuracy': max(history.history['val_accuracy']),
        'epochs_trained': len(history.history['accuracy']),

        # Percorso del modello salvato
        'model_path': 'best_model_Esercizio10.keras'
    }

    # Salva in formato pickle
    with open(filepath, 'wb') as f:
        pickle.dump(model_params, f)

    print(f"✅ Parametri salvati in: {filepath}")
    print(f"   Classi: {model_params['class_names']}")
    print(f"   Miglior accuratezza: {model_params['best_val_accuracy']:.4f}")

    # Salva anche il modello completo
    model.save('best_model_Esercizio10.keras')
    print(f"✅ Modello salvato in: best_model_Esercizio10.keras")

    return model_params


def load_model_params(filepath='classi.pkl'):
    """
    Carica i parametri salvati dal file classi.pkl.
    Utile per riutilizzare il modello senza riaddestrarlo.
    """
    with open(filepath, 'rb') as f:
        params = pickle.load(f)

    print(f"\n📂 Parametri caricati da: {filepath}")
    print(f"   Classi: {params['class_names']}")
    print(f"   Accuratezza: {params['best_val_accuracy']:.4f}")

    return params


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 8: METRICHE DI VALUTAZIONE E GRAFICI                              ║
# ║                                                                            ║
# ║ Metriche calcolate:                                                        ║
# ║ - Accuracy: % di predizioni corrette sul totale                            ║
# ║ - Precision: quante delle predizioni positive sono realmente positive      ║
# ║ - Recall: quanti dei positivi reali sono stati trovati                     ║
# ║ - F1-Score: media armonica di precision e recall                           ║
# ║ - Matrice di confusione: mostra errori per classe                          ║
# ║ - Curva ROC e AUC: misura la qualità della classificazione                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def evaluate_model(model, X_test, y_test, class_names, history):
    """
    Valuta il modello con metriche complete e crea grafici.

    SPIEGAZIONE METRICHE:

    ACCURACY (Accuratezza):
    - Formula: (Predizioni Corrette) / (Totale Predizioni)
    - "Su 100 foto, quante ne classifica giusto?"

    PRECISION (Precisione):
    - Formula: Veri Positivi / (Veri Positivi + Falsi Positivi)
    - "Quando dice 'cappello', quante volte ha ragione?"
    - Alta precision = pochi falsi allarmi

    RECALL (Richiamo / Sensibilità):
    - Formula: Veri Positivi / (Veri Positivi + Falsi Negativi)
    - "Quanti cappelli riesce a trovare su tutti quelli presenti?"
    - Alto recall = trova quasi tutti gli oggetti

    F1-SCORE:
    - Formula: 2 × (Precision × Recall) / (Precision + Recall)
    - Media armonica: bilancia precision e recall
    - Utile quando le classi sono sbilanciate

    MATRICE DI CONFUSIONE:
    - Mostra quanti oggetti di ogni classe sono stati predetti correttamente/erroneamente
    - Diagonale: predizioni corrette
    - Fuori diagonale: errori

    CURVA ROC:
    - Grafico che mostra il trade-off tra True Positive Rate e False Positive Rate
    - AUC (Area Under Curve): più vicino a 1 = migliore
    """

    print("\n" + "=" * 60)
    print("VALUTAZIONE DEL MODELLO")
    print("=" * 60)

    # --- PREDIZIONI ---
    # Il modello produce probabilità (0-1), convertiamo in classi (0 o 1)
    y_pred_proba = model.predict(X_test, verbose=0)  # Probabilità
    y_pred = (y_pred_proba > 0.5).astype(int).flatten()  # Classi (soglia 0.5)

    # --- METRICHE NUMERICHE ---
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, average=None
    )

    print(f"\n📊 METRICHE DI VALUTAZIONE SUL TEST SET:")
    print(f"{'─' * 50}")
    print(f"   Accuratezza complessiva: {accuracy:.4f} ({accuracy*100:.1f}%)")
    print()

    # Stampa metriche per ogni classe
    for i, name in enumerate(class_names):
        print(f"   Classe '{name}':")
        print(f"      Precision: {precision[i]:.4f} ({precision[i]*100:.1f}%)")
        print(f"      Recall:    {recall[i]:.4f} ({recall[i]*100:.1f}%)")
        print(f"      F1-Score:  {f1[i]:.4f} ({f1[i]*100:.1f}%)")
        print(f"      Support:   {support[i]} campioni")
        print()

    # Report completo sklearn
    print(f"\n📋 CLASSIFICATION REPORT:")
    print(f"{'─' * 50}")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # --- MATRICE DI CONFUSIONE ---
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n📊 MATRICE DI CONFUSIONE:")
    print(f"{'─' * 50}")
    print(f"                  Predetto")
    print(f"                  {'Cappello':>10} {'Scarpa':>10}")
    print(f"   Reale Cappello {cm[0][0]:>10} {cm[0][1]:>10}")
    print(f"   Reale Scarpa   {cm[1][0]:>10} {cm[1][1]:>10}")

    # SOSPETTO: Verifica classi sbilanciate nella matrice
    print(f"\n🔍 ANALISI BILANCIAMENTO CLASSI:")
    print(f"   Cappelli nel test set: {np.sum(y_test == 0)}")
    print(f"   Scarpe nel test set:   {np.sum(y_test == 1)}")
    if np.sum(y_test == 0) != np.sum(y_test == 1):
        print(f"   ⚠️  Le classi NON sono bilanciate nel test set!")
        print(f"   → Questo può influenzare le metriche: guardare F1-score per classe")
    else:
        print(f"   ✅ Le classi sono bilanciate nel test set")

    # ╔═══════════════════════════════════════════╗
    # ║ CREAZIONE GRAFICI DI VALUTAZIONE          ║
    # ╚═══════════════════════════════════════════╝

    # Crea una figura con 4 sottografici (2x2)
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle('Esercizio 10 - Metriche di Valutazione\nClassificazione Cappello vs Scarpa',
                 fontsize=16, fontweight='bold', y=1.02)

    # --- GRAFICO 1: Accuratezza durante l'addestramento ---
    # Mostra come l'accuratezza cambia epoca per epoca
    ax1 = axes[0, 0]
    ax1.plot(history.history['accuracy'], label='Training', linewidth=2, color='#2196F3')
    ax1.plot(history.history['val_accuracy'], label='Validazione', linewidth=2, color='#FF5722')
    ax1.axhline(y=0.8, color='green', linestyle='--', alpha=0.7, label='Soglia 80%')
    ax1.set_title('Accuratezza per Epoca', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoca')
    ax1.set_ylabel('Accuratezza')
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1.05])

    # --- GRAFICO 2: Loss durante l'addestramento ---
    # La loss deve SCENDERE. Se sale sulla validazione → overfitting
    ax2 = axes[0, 1]
    ax2.plot(history.history['loss'], label='Training', linewidth=2, color='#2196F3')
    ax2.plot(history.history['val_loss'], label='Validazione', linewidth=2, color='#FF5722')
    ax2.set_title('Loss per Epoca', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoca')
    ax2.set_ylabel('Loss (Binary Crossentropy)')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    # --- GRAFICO 3: Matrice di Confusione (Heatmap) ---
    ax3 = axes[1, 0]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax3, square=True, linewidths=1, linecolor='white',
                annot_kws={'size': 18, 'fontweight': 'bold'})
    ax3.set_title('Matrice di Confusione', fontsize=13, fontweight='bold')
    ax3.set_xlabel('Classe Predetta', fontsize=11)
    ax3.set_ylabel('Classe Reale', fontsize=11)

    # --- GRAFICO 4: Curva ROC ---
    # ROC = Receiver Operating Characteristic
    # Mostra la qualità del classificatore a diverse soglie
    ax4 = axes[1, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    ax4.plot(fpr, tpr, color='#2196F3', linewidth=2,
             label=f'ROC curve (AUC = {roc_auc:.3f})')
    ax4.plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.5,
             label='Classificatore casuale')
    ax4.fill_between(fpr, tpr, alpha=0.1, color='#2196F3')
    ax4.set_title('Curva ROC', fontsize=13, fontweight='bold')
    ax4.set_xlabel('False Positive Rate')
    ax4.set_ylabel('True Positive Rate')
    ax4.legend(loc='lower right')
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim([0, 1])
    ax4.set_ylim([0, 1.05])

    plt.tight_layout()
    plt.savefig('Esercizio10_metriche_valutazione.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✅ Grafici salvati in: Esercizio10_metriche_valutazione.png")

    # --- GRAFICO AGGIUNTIVO: Metriche per classe (barre) ---
    fig2, ax5 = plt.subplots(figsize=(10, 5))
    x = np.arange(len(class_names))
    width = 0.25

    bars1 = ax5.bar(x - width, precision, width, label='Precision', color='#2196F3', alpha=0.8)
    bars2 = ax5.bar(x, recall, width, label='Recall', color='#FF5722', alpha=0.8)
    bars3 = ax5.bar(x + width, f1, width, label='F1-Score', color='#4CAF50', alpha=0.8)

    ax5.set_xlabel('Classe')
    ax5.set_ylabel('Score')
    ax5.set_title('Esercizio 10 - Precision, Recall, F1-Score per Classe',
                  fontsize=13, fontweight='bold')
    ax5.set_xticks(x)
    ax5.set_xticklabels(class_names)
    ax5.legend()
    ax5.set_ylim([0, 1.15])
    ax5.grid(True, alpha=0.3, axis='y')

    # Aggiunge valori sopra le barre
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax5.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points", ha='center', fontsize=10)

    plt.tight_layout()
    plt.savefig('Esercizio10_metriche_per_classe.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Grafico metriche per classe salvato")

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'roc_auc': roc_auc
    }


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 9: PREDIZIONE SU NUOVE IMMAGINI                                   ║
# ║                                                                            ║
# ║ Funzione per classificare una nuova immagine usando il modello addestrato  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def predict_image(model, image_path, class_names):
    """
    Classifica una singola immagine usando il modello addestrato.

    Parametri:
    ----------
    model : modello Keras addestrato
    image_path : percorso dell'immagine da classificare
    class_names : lista con i nomi delle classi

    Ritorna:
    --------
    dict con classe predetta, probabilità e confidenza
    """
    # Carica e preprocessa l'immagine
    img = load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    # Predizione
    prediction = model.predict(img_array, verbose=0)[0][0]

    # Interpreta il risultato
    if prediction > 0.5:
        predicted_class = class_names[1]  # scarpa
        confidence = prediction
    else:
        predicted_class = class_names[0]  # cappello
        confidence = 1 - prediction

    print(f"\n🔮 Predizione per: {image_path}")
    print(f"   Classe: {predicted_class}")
    print(f"   Confidenza: {confidence:.2%}")
    print(f"   Probabilità scarpa: {prediction:.4f}")
    print(f"   Probabilità cappello: {1-prediction:.4f}")

    return {
        'class': predicted_class,
        'confidence': confidence,
        'probability_scarpa': prediction,
        'probability_cappello': 1 - prediction
    }


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ SEZIONE 10: ESECUZIONE PRINCIPALE (MAIN)                                  ║
# ║                                                                            ║
# ║ Orchestrazione di tutto il processo:                                       ║
# ║ 1. Data Augmentation → 2. Caricamento → 3. Training → 4. Valutazione     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

if __name__ == '__main__':

    print("╔" + "═" * 58 + "╗")
    print("║  ESERCIZIO 10 - Classificazione Cappello vs Scarpa       ║")
    print("║  Transfer Learning con MobileNetV2                       ║")
    print("║  Inizio: 12:00                                           ║")
    print("╚" + "═" * 58 + "╝")

    # ================================================================
    # STEP 1: DATA AUGMENTATION
    # Crea 250 immagini per classe partendo da una singola foto
    # ================================================================

    # NOTA: Le immagini originali devono essere nella cartella corrente
    # Su Google Colab, caricarle con:
    #   from google.colab import files
    #   uploaded = files.upload()

    # Percorsi delle immagini originali
    cappello_source = 'image augmentation cappello.jpg'
    scarpa_source = 'image augmentation scarpa.jpg'

    # Verifica che le immagini esistano
    for src in [cappello_source, scarpa_source]:
        if not os.path.exists(src):
            print(f"⚠️  ATTENZIONE: File non trovato: {src}")
            print(f"   Su Google Colab, carica il file con:")
            print(f"   from google.colab import files")
            print(f"   uploaded = files.upload()")
            print(f"   Oppure trascinalo nella barra laterale dei file")

    # Genera immagini augmentate
    if os.path.exists(cappello_source):
        n_cappello = create_augmented_images(
            cappello_source, DATASET_DIR, 'cappello', NUM_AUGMENTED_IMAGES
        )
    else:
        print("⚠️  Salto data augmentation per cappello (file non trovato)")
        n_cappello = 0

    if os.path.exists(scarpa_source):
        n_scarpa = create_augmented_images(
            scarpa_source, DATASET_DIR, 'scarpa', NUM_AUGMENTED_IMAGES
        )
    else:
        print("⚠️  Salto data augmentation per scarpa (file non trovato)")
        n_scarpa = 0

    # ================================================================
    # STEP 2: CARICAMENTO E PREPARAZIONE DATASET
    # ================================================================

    X, y = load_dataset(DATASET_DIR)

    if len(X) == 0:
        print("\n❌ ERRORE: Nessuna immagine caricata! Verifica i file sorgente.")
        exit()

    # Divisione in Training (80%) e Test (20%)
    # random_state=42: seme casuale per riproducibilità
    # stratify=y: mantiene la stessa proporzione di classi in train e test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=VALIDATION_SPLIT,    # 20% per il test
        random_state=42,               # Riproducibilità
        stratify=y                     # Mantiene bilanciamento classi
    )

    print(f"\n📊 Divisione dataset:")
    print(f"   Training:   {len(X_train)} immagini")
    print(f"   Test:       {len(X_test)} immagini")
    print(f"   Train cappelli: {np.sum(y_train == 0)}, scarpe: {np.sum(y_train == 1)}")
    print(f"   Test cappelli:  {np.sum(y_test == 0)}, scarpe: {np.sum(y_test == 1)}")

    # ================================================================
    # STEP 3: BILANCIAMENTO CLASSI
    # SOSPETTO: le classi potrebbero non avere lo stesso numero di oggetti
    # ================================================================

    # Calcola i pesi per bilanciare le classi
    # Se una classe ha meno campioni, avrà un peso maggiore
    n_class_0 = np.sum(y_train == 0)  # Numero cappelli
    n_class_1 = np.sum(y_train == 1)  # Numero scarpe
    total = n_class_0 + n_class_1

    class_weight = {
        0: total / (2 * n_class_0),  # Peso per cappelli
        1: total / (2 * n_class_1),  # Peso per scarpe
    }

    print(f"\n⚖️  Class weights calcolati:")
    print(f"   Cappello (0): {class_weight[0]:.4f}")
    print(f"   Scarpa   (1): {class_weight[1]:.4f}")
    if abs(class_weight[0] - class_weight[1]) > 0.1:
        print(f"   → Classi sbilanciate! I pesi compensano la differenza")
    else:
        print(f"   → Classi bilanciate, pesi quasi uguali")

    # ================================================================
    # STEP 4: COSTRUZIONE E ADDESTRAMENTO DEL MODELLO
    # ================================================================

    model = build_model()

    history = train_model(
        model, X_train, y_train, X_test, y_test,
        class_weight=class_weight
    )

    # ================================================================
    # STEP 5: SALVATAGGIO PARAMETRI IN classi.pkl
    # ================================================================

    model_params = save_model_params(model, history, CLASS_NAMES, MODEL_SAVE_PATH)

    # ================================================================
    # STEP 6: VALUTAZIONE CON METRICHE E GRAFICI
    # ================================================================

    metrics = evaluate_model(model, X_test, y_test, CLASS_NAMES, history)

    # ================================================================
    # STEP 7: VERIFICA CARICAMENTO classi.pkl
    # ================================================================

    print("\n" + "=" * 60)
    print("VERIFICA CARICAMENTO classi.pkl")
    print("=" * 60)
    loaded_params = load_model_params(MODEL_SAVE_PATH)
    print(f"\n   Contenuto file classi.pkl:")
    for key, value in loaded_params.items():
        if key != 'training_history':  # Non stampare lo storico completo
            print(f"   - {key}: {value}")

    # ================================================================
    # STEP 8: RIEPILOGO FINALE
    # ================================================================

    print("\n" + "╔" + "═" * 58 + "╗")
    print("║  RIEPILOGO FINALE - ESERCIZIO 10                         ║")
    print("╠" + "═" * 58 + "╣")
    print(f"║  Accuratezza: {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.1f}%){'':>28}║")
    print(f"║  ROC AUC: {metrics['roc_auc']:.4f}{'':>40}║")
    print(f"║  Soglia richiesta: 80%{'':>32}║")
    if metrics['accuracy'] >= 0.80:
        print(f"║  ✅ OBIETTIVO RAGGIUNTO!{'':>31}║")
    else:
        print(f"║  ⚠️  Sotto soglia - ottimizzare parametri{'':>13}║")
    print(f"║{'':>55}║")
    print(f"║  File salvati:{'':>40}║")
    print(f"║  - classi.pkl (parametri){'':>29}║")
    print(f"║  - best_model_Esercizio10.keras (modello){'':>13}║")
    print(f"║  - Esercizio10_metriche_valutazione.png{'':>15}║")
    print(f"║  - Esercizio10_metriche_per_classe.png{'':>16}║")
    print(f"║{'':>55}║")
    print(f"║  Fine: 12:45{'':>42}║")
    print("╚" + "═" * 58 + "╝")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ APPENDICE: ISTRUZIONI PER GOOGLE COLAB                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
"""
COME ESEGUIRE SU GOOGLE COLAB:
═══════════════════════════════

1. Aprire Google Colab: https://colab.research.google.com
2. Runtime → Change runtime type → GPU (T4 o superiore)
3. Caricare le immagini originali:
   - "image augmentation cappello.jpg"
   - "image augmentation scarpa.jpg"

   Metodo 1 (upload manuale):
   ```python
   from google.colab import files
   uploaded = files.upload()  # Seleziona i 2 file
   ```

   Metodo 2 (trascinamento):
   - Aprire il pannello File (icona cartella a sinistra)
   - Trascinare i file nella cartella /content/

4. Copiare tutto il codice in una cella e eseguire (Shift+Enter)
5. L'addestramento richiede circa 5-10 minuti con GPU T4
6. I grafici appariranno automaticamente
7. Scaricare i file generati:
   ```python
   from google.colab import files
   files.download('classi.pkl')
   files.download('best_model_Esercizio10.keras')
   files.download('Esercizio10_metriche_valutazione.png')
   ```

TROUBLESHOOTING:
═══════════════
- "GPU non trovata": Runtime → Change runtime type → GPU
- "File non trovato": Verifica che le immagini siano caricate nella cartella corrente
- "Out of Memory": Ridurre BATCH_SIZE a 8 o NUM_AUGMENTED_IMAGES a 200
- "Accuratezza bassa": Aumentare NUM_AUGMENTED_IMAGES a 400 o EPOCHS a 100
"""

SETUP AMBIENTE
TensorFlow versione: 2.19.0
GPU disponibili: []
⚠️  Nessuna GPU trovata. Su Colab: Runtime → Change runtime type → GPU

CONFIGURAZIONE PARAMETRI
Dimensione immagini: 224x224x3
Immagini augmentate per classe: 250
Batch size: 16
Epoche massime: 50
Learning rate: 0.0001
Neuroni Dense layer: 128
Dropout rates: 0.3, 0.2
Early stopping patience: 10

╔══════════════════════════════════════════════════════════╗
║  ESERCIZIO 10 - Classificazione Cappello vs Scarpa       ║
║  Transfer Learning con MobileNetV2                       ║
║  Inizio: 12:00                                           ║
╚══════════════════════════════════════════════════════════╝
⚠️  ATTENZIONE: File non trovato: image augmentation cappello.jpg
   Su Google Colab, carica il file con:
   from google.colab import files
   uploaded = files.upload()
   Oppure trascinalo nella barra laterale dei file
⚠️  Salto data augmentation per cappello (file non trovato)

📸 Caricamento immagine: image augmentation scarpa.jpg

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


ADDESTRAMENTO DEL MODELLO

🏋️ Inizio addestramento...
   Training samples: 197
   Validation samples: 50
   Batch size: 16
   Epoche massime: 50
   Early stopping patience: 10
   Class weights: {0: np.float64(inf), 1: np.float64(0.5)}

Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 566ms/step - accuracy: 0.9478 - loss: 0.1181
Epoch 1: val_accuracy improved from -inf to 1.00000, saving model to best_model_Esercizio10.keras
13/13 ━━━━━━━━━━━━━━━━━━━━ 19s 935ms/step - accuracy: 0.9501 - loss: 0.1142 - val_accuracy: 1.0000 - val_loss: 0.0092 - learning_rate: 1.0000e-04
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 678ms/step - accuracy: 1.0000 - loss: 0.0057
Epoch 2: val_accuracy did not improve from 1.00000
13/13 ━━━━━━━━━━━━━━━━━━━━ 20s 896ms/step - accuracy: 1.0000 - loss: 0.0056 - val_accuracy: 1.0000 - val_loss: 0.0016 - learning_rate: 1.0000e-04
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 674ms/step - accuracy: 1.0000 - loss: 0.0012
Epoch 3: val_accuracy did not improve from 1.00000
13/13 ━━━━━━━

KeyboardInterrupt: 